# 365 Probabilidades - Dia #131
## Qual a probabilidade de uma ação sorteada perder para a renda fixa?

**Tipo:** Risco
**Data de publicação:** 2026-10-22
**Ferramenta:** Python
**Decisão analisada:** Uma ação ou o Tesouro?
**Hashtag:** #365Probabilidades #Dia131

---

### 📖 A História

Todo mundo tem uma ação assim. A que o amigo falou, a que apareceu na reportagem, a que
"ia explodir". Você comprou um pouco, só para ver. Faz uns anos. Ela não explodiu. Ou
explodiu e você vendeu antes.

A frase que fez você comprar é a frase que o mercado inteiro repete: ação rende mais do
que renda fixa no longo prazo. Para a bolsa inteira, a frase é verdadeira. Fui ver se
vale para uma ação.

---

### 📚 O Conceito

A bolsa como um todo rende mais do que o título mais seguro do governo. Isso está medido
em um século de dados e não é a pergunta de hoje. A pergunta é sobre a ação típica, a
que você sorteia, e a resposta é diferente por causa da **assimetria**.

O retorno de uma ação não é simétrico. O máximo que ela perde é 100%. O máximo que ela
ganha não tem teto. Quando uma distribuição tem cauda longa para a direita, a média fica
acima da mediana, e quanto mais tempo passa, mais os juros compostos esticam a cauda. O
resultado é um mercado em que a média sobe e a ação do meio não acompanha. A média da
bolsa é feita por pouquíssimas empresas; a ação típica é outra coisa.

A comparação do paper é com o título do Tesouro americano de um mês, o ativo mais seguro
que existe em dólar. No Brasil, o equivalente é o Tesouro Selic. É isso que "renda fixa"
quer dizer aqui.

---

### 🧮 O Modelo

**Fonte primária:**
- **Bessembinder, H. (2018).** Do stocks outperform Treasury bills? *Journal of Financial
  Economics*, 129(3), 440-457. Base CRSP, todas as ações ordinárias de NYSE, Amex e
  Nasdaq, julho de 1926 a dezembro de 2016. **N = 25.967 ações.** Retorno buy-and-hold
  da listagem até a saída da bolsa, comparado com o Tesouro de um mês no mesmo período.
  Contagem exaustiva, não amostra.
- **Bessembinder, Chen, Choi & Wei (2023).** Long-term shareholder returns: evidence from
  64,000 global stocks. *Financial Analysts Journal*, 79(3). 42 países, 1990 a 2020.
  Entra como réplica fora dos Estados Unidos.

**Modelo próprio:**
- Posterior Beta (Jeffreys) da fração que perde, para mostrar o tamanho do intervalo com
  N = 25.967.
- Um modelo estilizado de retorno mensal, fator de mercado mais ruído próprio de cada
  ação, calibrado em três números do paper (ação sorteada ganha do Tesouro em 47,8% das
  décadas; carteira sorteada de 25 ações, 86,9%; de 100, 93,1%). Serve para duas coisas:
  mostrar a assimetria (média acima do Tesouro, mediana abaixo) e desenhar a escada da
  diversificação, de 1 a 100 ações, por Monte Carlo (semente 42).
- A conta de quantas ações são precisas para carregar uma das 4% que fizeram o ganho,
  pela hipergeométrica.
- Sem ×0,80: o dado é contagem de registros de preço, nada autorrelatado. O ×0,80 é
  regra deste projeto para proporções de survey.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

VERMELHO, DOURADO, VERDE, CINZA = '#c0392b', '#c8a84b', '#1a5f5a', '#6b6a64'
SEMENTE = 42
ger = np.random.default_rng(SEMENTE)

def vir(x, casas=1):
    return f"{x:.{casas}f}".replace(".", ",")

def pct(x, casas=1):
    return vir(100 * x, casas) + "%"

def mil(n):
    return f"{int(round(n)):,}".replace(",", ".")

def beta_j(k, n):
    d = stats.beta(k + 0.5, n - k + 0.5)
    return d.mean(), d.interval(0.95)

print("✅ Bibliotecas carregadas")


In [ ]:
# --- DADOS DA LITERATURA ---

# Bessembinder (2018), JFE 129(3). CRSP, julho/1926 a dezembro/2016.
N_ACOES = 25967                 # ações ordinárias na base
P_GANHA_VIDA = 0.426            # retorno da vida inteira acima do Tesouro de 1 mês
P_PERDE_VIDA = 1 - P_GANHA_VIDA # 57,4%
VIDA_MEDIANA_ANOS = 7.5         # tempo mediano de uma ação na bolsa

# Fração de ações sorteadas que ganham do Tesouro, por horizonte
HORIZONTES = {"um mês": 0.478, "um ano": 0.516, "uma década": 0.495, "a vida inteira": 0.426}

# Simulações do próprio paper: carteira sorteada, mantida 10 anos, rebalanceada todo mês
ESCADA_PAPER = {1: 0.478, 25: 0.869, 100: 0.931}

# Concentração do ganho líquido (riqueza criada acima do Tesouro), 1926-2016
N_GRAFICO_RIQUEZA = 25332       # ações com dado de riqueza no paper
N_TODO_GANHO = 1092             # empresas que explicam todo o ganho líquido
N_METADE = 90                   # empresas que explicam mais da metade
N_TOP5, FRAC_TOP5 = 5, 0.10     # as 5 maiores, 10% do total
RIQUEZA_TRI = 35                # cerca de US$ 35 trilhões criados acima do Tesouro
FRAC_TODO_GANHO = N_TODO_GANHO / N_GRAFICO_RIQUEZA

# Réplica global, Bessembinder, Chen, Choi & Wei (2023), FAJ. 42 países, 1990-2020.
N_GLOBAL = 64000
P_PERDE_EUA_90_20 = 0.552
P_PERDE_FORA_90_20 = 0.574
FRAC_TODO_GANHO_GLOBAL = 0.024

print("=" * 70)
print("  DADOS DA LITERATURA — AÇÃO SORTEADA × TESOURO")
print("=" * 70)
print(f"\n  Bessembinder (2018), CRSP 1926-2016, N = {mil(N_ACOES)} ações")
print(f"  → Ganham do Tesouro na vida inteira:     {pct(P_GANHA_VIDA)}")
print(f"  → Perdem para o Tesouro na vida inteira: {pct(P_PERDE_VIDA)}  (quatro em sete)")
print(f"  → Vida mediana de uma ação na bolsa:     {vir(VIDA_MEDIANA_ANOS)} anos")
print("\n  Fração que ganha do Tesouro, por horizonte:")
for h, p in HORIZONTES.items():
    print(f"  → {h:<16s} {pct(p)}")
print("\n  Carteira sorteada, 10 anos (simulação do paper):")
for n, p in ESCADA_PAPER.items():
    print(f"  → {n:>3d} ações: ganha do Tesouro em {pct(p)} das carteiras")
print(f"\n  Concentração do ganho líquido (US$ {RIQUEZA_TRI} tri, 1926-2016):")
print(f"  → {N_TOP5} empresas:     {pct(FRAC_TOP5, 0)} do ganho")
print(f"  → {N_METADE} empresas:    mais da metade")
print(f"  → {mil(N_TODO_GANHO)} empresas: todo o ganho líquido ({pct(FRAC_TODO_GANHO)} das ações)")
print(f"  → as outras {pct(1 - FRAC_TODO_GANHO)}, juntas, empataram com o Tesouro")
print(f"\n  Réplica global (2023), {mil(N_GLOBAL)} ações, 42 países, 1990-2020:")
print(f"  → Perdem para o Tesouro, EUA:            {pct(P_PERDE_EUA_90_20)}")
print(f"  → Perdem para o Tesouro, fora dos EUA:   {pct(P_PERDE_FORA_90_20)}")
print(f"  → Empresas com todo o ganho líquido:     {pct(FRAC_TODO_GANHO_GLOBAL)}")
print("=" * 70)


In [ ]:
# --- O MODELO ---

# 1. O tamanho do intervalo com N = 25.967
k_perde = round(P_PERDE_VIDA * N_ACOES)
m_perde, ic_perde = beta_j(k_perde, N_ACOES)
quatro_em_sete = 4 / 7

# 2. Modelo estilizado: log-retorno mensal = mu + fator de mercado + ruído próprio
MESES_DECADA = 120

def p_uma_acao(mu, sm, si, meses=MESES_DECADA):
    # uma ação, buy-and-hold: a soma dos log-retornos é normal
    st = np.sqrt(sm ** 2 + si ** 2)
    return stats.norm.cdf(np.sqrt(meses) * mu / st)

def mu_para(p, sm, si, meses=MESES_DECADA):
    # o mu que faz uma ação sorteada ganhar do Tesouro em p das décadas
    return stats.norm.ppf(p) * np.sqrt(sm ** 2 + si ** 2) / np.sqrt(meses)

def simula_carteira(n, mu, sm, si, sims, meses=MESES_DECADA, chunk=500):
    # carteira de n ações sorteadas, pesos iguais, rebalanceada todo mês; devolve o fator de crescimento
    saida = []
    for s0 in range(0, sims, chunk):
        k = min(chunk, sims - s0)
        f = ger.normal(0, sm, (k, meses, 1))
        e = ger.normal(0, si, (k, meses, n))
        r_cart = (np.exp(mu + f + e) - 1).mean(axis=2)
        saida.append(np.prod(1 + r_cart, axis=1))
    return np.concatenate(saida)

# calibração: mu amarrado ao 47,8% de uma ação; sm e si escolhidos pelo 86,9% (25) e 93,1% (100)
grade = []
for sm in [0.030, 0.035, 0.040, 0.045]:
    for si in [0.09, 0.10, 0.11, 0.12, 0.13]:
        mu = mu_para(ESCADA_PAPER[1], sm, si)
        p25 = (simula_carteira(25, mu, sm, si, 4000) > 1).mean()
        p100 = (simula_carteira(100, mu, sm, si, 2000) > 1).mean()
        grade.append((sm, si, mu, p25, p100, (p25 - ESCADA_PAPER[25]) ** 2 + (p100 - ESCADA_PAPER[100]) ** 2))
grade = pd.DataFrame(grade, columns=["sm", "si", "mu", "p25", "p100", "erro"])
melhor = grade.loc[grade["erro"].idxmin()]
SM, SI, MU = float(melhor["sm"]), float(melhor["si"]), float(melhor["mu"])
ST = np.sqrt(SM ** 2 + SI ** 2)
media_mensal = np.exp(MU + ST ** 2 / 2) - 1     # retorno aritmético médio acima do Tesouro
mediana_mensal = np.exp(MU) - 1

# 3. A assimetria: uma ação, pela vida mediana de 7,5 anos (90 meses)
MESES_VIDA = int(round(VIDA_MEDIANA_ANOS * 12))
SIMS = 20000
g_vida = simula_carteira(1, MU, SM, SI, SIMS, meses=MESES_VIDA)
p_vida_modelo = (g_vida > 1).mean()
mediana_vida, media_vida = np.median(g_vida) - 1, g_vida.mean() - 1
p_dobra = (g_vida > 2).mean(); p_perde_metade = (g_vida < 0.5).mean()

# 4. A escada da diversificação, 10 anos
NS = [1, 2, 3, 5, 10, 25, 50, 100]
escada = []
for n in NS:
    g = simula_carteira(n, MU, SM, SI, SIMS if n <= 25 else 8000)
    k = int((g > 1).sum()); m, ic = beta_j(k, len(g))
    escada.append({"n": n, "p": m, "lo": ic[0], "hi": ic[1], "paper": ESCADA_PAPER.get(n, np.nan)})
escada = pd.DataFrame(escada)

# 5. Quantas ações para carregar uma das 1.092 (hipergeométrica, sorteio sem reposição)
def p_contem_vencedora(n, K=N_TODO_GANHO, N=N_GRAFICO_RIQUEZA):
    return 1 - stats.hypergeom(N, K, n).pmf(0)
contem = {n: p_contem_vencedora(n) for n in NS}

print("=" * 70)
print("  MODELO — AÇÃO SORTEADA × TESOURO")
print("=" * 70)
print(f"\n  1. A fração que perde na vida inteira: {pct(P_PERDE_VIDA)}, N = {mil(N_ACOES)}")
print(f"     posterior Beta (Jeffreys): {pct(m_perde)}  IC 95% [{pct(ic_perde[0])}; {pct(ic_perde[1])}]")
print(f"     quatro em sete = {pct(quatro_em_sete)}; o intervalo tem {vir((ic_perde[1]-ic_perde[0])*100)} ponto de largura")
print(f"\n  2. Modelo estilizado calibrado no paper (grade de {len(grade)} combinações):")
print(f"     fator de mercado {pct(SM)}/mês, ruído próprio {pct(SI)}/mês, mu = {vir(MU*100, 3)}%/mês")
print(f"     média mensal acima do Tesouro: {vir(media_mensal*100, 2)}%  |  mediana mensal: {vir(mediana_mensal*100, 2)}%")
print(f"     reproduz: 25 ações {pct(melhor['p25'])} (paper {pct(ESCADA_PAPER[25])}); 100 ações {pct(melhor['p100'])} (paper {pct(ESCADA_PAPER[100])})")
print(f"\n  3. Uma ação sorteada pela vida mediana ({MESES_VIDA} meses), {mil(SIMS)} simulações:")
print(f"     ganha do Tesouro em {pct(p_vida_modelo)} (o dado real é {pct(P_GANHA_VIDA)}: o modelo não tem falência)")
print(f"     retorno mediano acima do Tesouro: {vir(mediana_vida*100)}%  |  retorno médio: +{vir(media_vida*100)}%")
print(f"     dobra o Tesouro em {pct(p_dobra)}; fica abaixo da metade em {pct(p_perde_metade)}")
print(f"\n  4. A escada da diversificação (10 anos, carteira sorteada, pesos iguais):")
for _, r in escada.iterrows():
    ref = f"   paper {pct(r['paper'])}" if not np.isnan(r["paper"]) else ""
    print(f"     {int(r['n']):>3d} {'ação ' if r['n'] == 1 else 'ações'}: ganha do Tesouro em {pct(r['p'])} [{pct(r['lo'])}; {pct(r['hi'])}]{ref}")
print(f"\n  5. Chance de a carteira sorteada conter ao menos uma das {mil(N_TODO_GANHO)} que fizeram o ganho:")
for n in NS:
    print(f"     {n:>3d} {'ação ' if n == 1 else 'ações'}: {pct(contem[n])}")
print("=" * 70)


In [ ]:
# --- VISUALIZAÇÃO ---

# ══ GRÁFICO 1 · os horizontes ══
fig1, ax1 = plt.subplots(figsize=(11, 6))
nomes = list(HORIZONTES.keys()) + ["EUA\n1990-2020", "fora dos EUA\n1990-2020"]
vals = [100 * v for v in HORIZONTES.values()] + [100 * (1 - P_PERDE_EUA_90_20), 100 * (1 - P_PERDE_FORA_90_20)]
cores = [CINZA, CINZA, CINZA, VERMELHO, DOURADO, DOURADO]
xs = np.arange(len(nomes))
ax1.bar(xs, vals, color=cores, alpha=0.92, width=0.62)
ax1.axhline(50, color="#333333", lw=1.3, ls="--")
ax1.text(len(nomes) - 0.5, 50.8, "cara ou coroa", ha="right", fontsize=10.5, color="#333333")
for x, v in zip(xs, vals):
    ax1.text(x, v + 1, vir(v) + "%", ha="center", fontsize=13, fontweight="bold", color="#222222",
             bbox=dict(facecolor="white", edgecolor="none", pad=1.5), zorder=6)
ax1.axvline(3.5, color="#bbbbbb", lw=1)
ax1.text(1.5, 64, "Estados Unidos, 1926-2016, 25.967 ações", ha="center", fontsize=10, color=CINZA)
ax1.text(4.5, 64, "réplica global, 2023", ha="center", fontsize=10, color="#8a6f1f")
ax1.set_xticks(xs); ax1.set_xticklabels(nomes, fontsize=10.5)
ax1.set_ylim(0, 68); ax1.set_ylabel("ações sorteadas que ganham do Tesouro (%)")
ax1.set_title(f"Uma ação sorteada ganha do Tesouro em metade dos meses e em {pct(P_GANHA_VIDA)} das vidas\n"
              "quanto mais longo o prazo, mais a assimetria aparece", fontsize=12.5, pad=14)
plt.figtext(0.5, 0.005, "Fonte: Bessembinder (2018), JFE 129(3); Bessembinder, Chen, Choi & Wei (2023), FAJ  |  #365Probabilidades",
            ha="center", fontsize=8.5, color="gray")
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig("dia-131-grafico-01-os-horizontes.png", dpi=150, bbox_inches="tight"); plt.close()
print("✅ Gráfico 1 salvo!")

# ══ GRÁFICO 2 · a assimetria ══
fig2, ax2 = plt.subplots(figsize=(11, 6))
lg = np.log10(g_vida)
bins = np.linspace(-1.6, 1.6, 81)
ax2.hist(lg[g_vida <= 1], bins=bins, color=VERMELHO, alpha=0.85, label=f"perde para o Tesouro: {pct(1 - p_vida_modelo)}")
ax2.hist(lg[g_vida > 1], bins=bins, color=VERDE, alpha=0.85, label=f"ganha do Tesouro: {pct(p_vida_modelo)}")
ax2.axvline(np.log10(1 + mediana_vida), color="#222222", lw=2.2)
ax2.axvline(np.log10(1 + media_vida), color=DOURADO, lw=2.2, ls="--")
ax2.text(np.log10(1 + mediana_vida) - 0.04, ax2.get_ylim()[1] * 0.93, f"mediana {vir(mediana_vida*100)}%", ha="right", fontsize=11, fontweight="bold", color="#222222")
ax2.text(np.log10(1 + media_vida) + 0.04, ax2.get_ylim()[1] * 0.93, f"média +{vir(media_vida*100)}%", ha="left", fontsize=11, fontweight="bold", color="#8a6f1f")
ticks = [0.1, 0.25, 0.5, 1, 2, 4, 8, 16]
ax2.set_xticks(np.log10(ticks)); ax2.set_xticklabels([f"{t:g}×" for t in ticks])
ax2.set_xlabel("quanto a ação rendeu em relação ao Tesouro, em 7,5 anos (escala log)"); ax2.set_ylabel("simulações")
ax2.legend(fontsize=10.5, loc="upper left")
ax2.set_title("A assimetria: a média fica acima do Tesouro, a ação do meio fica abaixo\n"
              f"modelo estilizado, uma ação sorteada pela vida mediana de {vir(VIDA_MEDIANA_ANOS)} anos, {mil(SIMS)} simulações", fontsize=12.5, pad=14)
plt.figtext(0.5, 0.005, "Modelo próprio calibrado em Bessembinder (2018); sem falência, por isso mais gentil que o dado real  |  #365Probabilidades",
            ha="center", fontsize=8.5, color="gray")
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig("dia-131-grafico-02-a-assimetria.png", dpi=150, bbox_inches="tight"); plt.close()
print("✅ Gráfico 2 salvo!")

# ══ GRÁFICO 3 · a escada ══
fig3, ax3 = plt.subplots(figsize=(11, 6))
ax3.fill_between(escada["n"], escada["lo"] * 100, escada["hi"] * 100, color=VERDE, alpha=0.15)
ax3.plot(escada["n"], escada["p"] * 100, marker="o", ms=8, lw=2.6, color=VERDE, label="modelo estilizado, carteira sorteada, 10 anos")
ax3.scatter(list(ESCADA_PAPER.keys()), [100 * v for v in ESCADA_PAPER.values()], s=160, color=DOURADO, zorder=6, marker="D", label="simulação do paper (1, 25 e 100 ações)")
ax3.plot(NS, [100 * contem[n] for n in NS], lw=1.6, ls="--", color=CINZA, label="chance de carregar uma das 4% que fizeram o ganho")
for _, r in escada.iterrows():
    ax3.text(r["n"] * 0.9, r["p"] * 100 + 3.2, pct(r["p"], 0), ha="right", fontsize=10.5, fontweight="bold", color=VERDE,
             bbox=dict(facecolor="white", edgecolor="none", pad=1.0), zorder=7)
ax3.axhline(50, color="#333333", lw=1.1, ls=":")
ax3.set_xscale("log"); ax3.set_xticks(NS); ax3.set_xticklabels([str(n) for n in NS])
ax3.set_xlim(0.85, 120); ax3.set_ylim(0, 104)
ax3.set_xlabel("ações na carteira (sorteadas, pesos iguais)"); ax3.set_ylabel("probabilidade (%)")
ax3.legend(fontsize=9.5, loc="lower right")
ax3.set_title(f"A escada da diversificação: de {pct(escada['p'].iloc[0], 0)} com uma ação a {pct(escada['p'].iloc[-1], 0)} com cem\n"
              "a mesma moeda viciada, jogada muitas vezes ao mesmo tempo", fontsize=12.5, pad=14)
plt.figtext(0.5, 0.005, "Modelo próprio, Monte Carlo, semente 42; losangos de Bessembinder (2018), Tabela de carteiras sorteadas  |  #365Probabilidades",
            ha="center", fontsize=8.5, color="gray")
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig("dia-131-grafico-03-a-escada.png", dpi=150, bbox_inches="tight"); plt.close()
print("✅ Gráfico 3 salvo!")

# ══ GRÁFICO 4 · a concentração ══
fig4, ax4 = plt.subplots(figsize=(11, 6))
pontos_n = [N_TOP5, N_METADE, N_TODO_GANHO, N_GRAFICO_RIQUEZA]
pontos_f = [FRAC_TOP5, 0.5, 1.0, 1.0]
rot = [f"{N_TOP5} empresas\n{pct(FRAC_TOP5, 0)} do ganho", f"{N_METADE} empresas\nmetade do ganho",
       f"{mil(N_TODO_GANHO)} empresas ({pct(FRAC_TODO_GANHO)})\ntodo o ganho líquido", f"as {mil(N_GRAFICO_RIQUEZA)}\nas outras 96% empataram\ncom o Tesouro"]
ax4.plot(pontos_n, [100 * f for f in pontos_f], marker="o", ms=10, lw=2.6, color=VERMELHO, drawstyle="default")
ax4.fill_between(pontos_n, 0, [100 * f for f in pontos_f], color=VERMELHO, alpha=0.08)
pos = [(N_TOP5 * 1.3, 100 * FRAC_TOP5 - 2, "left", "top"), (N_METADE * 1.3, 50 - 2, "left", "top"),
       (N_TODO_GANHO, 104, "center", "bottom"), (N_GRAFICO_RIQUEZA, 95, "right", "top")]
for (x, y, ha, va), t in zip(pos, rot):
    ax4.text(x, y, t, fontsize=10, color="#222222", ha=ha, va=va,
             bbox=dict(facecolor="white", edgecolor="none", pad=1.0), zorder=7)
ax4.set_xscale("log"); ax4.set_xlim(3, 40000); ax4.set_ylim(0, 118)
ax4.set_xticks([5, 10, 100, 1000, 10000, 25332]); ax4.set_xticklabels(["5", "10", "100", "1.000", "10.000", "25.332"])
ax4.set_xlabel("empresas, da que mais criou riqueza para a que menos (escala log)")
ax4.set_ylabel(f"parte do ganho líquido de 90 anos (%)")
ax4.set_title(f"{pct(FRAC_TODO_GANHO)} das empresas explicam todo o ganho líquido da bolsa americana desde 1926\n"
              f"cerca de US$ {RIQUEZA_TRI} trilhões acima do Tesouro; {N_METADE} empresas fizeram a metade", fontsize=12.5, pad=14)
plt.figtext(0.5, 0.005, "Fonte: Bessembinder (2018), JFE 129(3), riqueza criada acima do Tesouro, 1926-2016  |  #365Probabilidades",
            ha="center", fontsize=8.5, color="gray")
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig("dia-131-grafico-04-a-concentracao.png", dpi=150, bbox_inches="tight"); plt.close()
print("✅ Gráfico 4 salvo!")


### 💡 O Insight

**Quatro em cada sete ações perderam para o Tesouro ao longo da vida. 57,4%, com N = 25.967.**

A frase do mercado é verdadeira para a bolsa inteira e falsa para a ação típica. As duas
coisas ao mesmo tempo, pelo mesmo motivo: a média é feita por pouquíssimas empresas.
Noventa fizeram metade do ganho de um século. Mil e noventa e duas, 4,3%, fizeram todo o
ganho líquido. As outras 96%, somadas, empataram com o título mais seguro do governo.

Em um mês, a ação sorteada é cara ou coroa. Em dez anos, ainda é cara ou coroa. Na vida
inteira, quatro em sete perdem. Não porque o mercado cai. Porque a cauda que puxa a
média fica com poucos.

A escada é a conta que falta. Uma ação sorteada ganha do Tesouro em 48% das décadas.
Vinte e cinco, em 87%. Cem, em 93%. A mesma moeda viciada, jogada muitas vezes ao mesmo
tempo.

*Qual é a ação que você comprou porque "ia explodir", e quanto do seu dinheiro está nela?*

---

### ⚠️ Limitações do Modelo

- O dado é americano, de 1926 a 2016. A réplica global de 2023 (64 mil ações, 42
  países, 1990 a 2020) encontrou a mesma forma, 55% a 57% abaixo do Tesouro e 2,4% das
  empresas com todo o ganho líquido. Não há contagem equivalente publicada para a B3.
- O Tesouro de um mês é o piso da renda fixa, e o período inclui décadas de juro real
  baixo. Com juro real alto, como o brasileiro, a barra para a ação é mais alta, não mais
  baixa.
- O modelo estilizado é ilustração calibrada, não reprodução. Ele não tem falência nem
  saída da bolsa; por isso dá 48% de vitória na vida mediana onde o dado real dá 42,6%.
  A escada do modelo reproduz os três pontos do paper com erro de 1 a 3 pontos.
- "Sorteada" é o pressuposto. Quem escolhe ações com informação não sorteia; o paper mede
  o que acontece com a ação típica, não com a ação escolhida.
- Nada aqui é recomendação. É a distribuição.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*
